# DEEP NEURAL NETWORKS - ASSIGNMENT 3: RNN vs TRANSFORMER FOR TIME SERIES

## Recurrent Neural Networks vs Transformers for Time Series Prediction

| Field | Value |
|-------|-------|
| **BITS ID** | 2025ae05144 |
| **Name** | Shubham Pardeshi |
| **Email** | 2025ae05144@wilp.bits-pilani.ac.in |
| **Date** | May 2026 |

In [ ]:
# Install dependencies
!pip install -q tensorflow scikit-learn matplotlib seaborn pandas numpy

In [ ]:
# Imports & Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time
import json
import math

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## PART 1: DATASET LOADING AND EXPLORATION

In [ ]:
# 1.1 Dataset Loading - Jena Climate (weather temperature)
zip_path = tf.keras.utils.get_file(
    origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip",
    fname="jena_climate_2009_2016.csv.zip",
    extract=True
)
csv_path = zip_path.replace(".zip", "")
df = pd.read_csv(csv_path)

# Subsample to hourly (every 6th row - original is 10-minute intervals)
df = df.iloc[::6].reset_index(drop=True)

# Extract temperature column (univariate forecasting)
temperature = df["T (degC)"].values.astype(np.float32)

# Dataset metadata
dataset_name = "Jena Climate"
dataset_source = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
n_samples = len(temperature)
n_features = 1
sequence_length = 30
prediction_horizon = 1
problem_type = "time_series_forecasting"

primary_metric = "MAE"
metric_justification = (
    "MAE is chosen as the primary metric because it provides an interpretable error "
    "in the same units as temperature (degrees Celsius), making it easy to assess "
    "practical forecasting quality without penalizing outliers disproportionately."
)

print("DATASET INFORMATION")
print(f"Dataset: {dataset_name}")
print(f"Source: {dataset_source}")
print(f"Total Samples: {n_samples}")
print(f"Number of Features: {n_features}")
print(f"Sequence Length: {sequence_length}")
print(f"Prediction Horizon: {prediction_horizon}")
print(f"Primary Metric: {primary_metric}")
print(f"Metric Justification: {metric_justification}")

In [ ]:
# 1.2 Data Exploration
print("Basic Statistics:")
print(f"  Min temperature:  {temperature.min():.2f} °C")
print(f"  Max temperature:  {temperature.max():.2f} °C")
print(f"  Mean temperature: {temperature.mean():.2f} °C")
print(f"  Std temperature:  {temperature.std():.2f} °C")
print(f"  Missing values:   {np.isnan(temperature).sum()}")

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full time series
axes[0].plot(temperature, linewidth=0.5)
axes[0].set_title("Jena Climate - Full Temperature Time Series (Hourly)")
axes[0].set_xlabel("Time Step (hours)")
axes[0].set_ylabel("Temperature (°C)")

# Zoomed-in: 7 days = 168 hours
axes[1].plot(temperature[:168], linewidth=1.0)
axes[1].set_title("Zoomed In - First 7 Days")
axes[1].set_xlabel("Time Step (hours)")
axes[1].set_ylabel("Temperature (°C)")

plt.tight_layout()
plt.show()

In [ ]:
# 1.3 Preprocessing & Temporal Split
# 90/10 temporal split (NO shuffling)
split_idx = int(len(temperature) * 0.9)
train_data = temperature[:split_idx]
test_data = temperature[split_idx:]

# StandardScaler fit on train only
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_data.reshape(-1, 1)).flatten()
test_scaled = scaler.transform(test_data.reshape(-1, 1)).flatten()

# Combine for sequence creation (but maintain temporal boundary)
all_scaled = np.concatenate([train_scaled, test_scaled])

def create_sequences(data, seq_length, pred_horizon=1):
    """Create sequences using a sliding window approach."""
    X, y = [], []
    for i in range(len(data) - seq_length - pred_horizon + 1):
        X.append(data[i : i + seq_length])
        y.append(data[i + seq_length + pred_horizon - 1])
    return np.array(X), np.array(y)

# Create sequences from train and test separately to avoid data leakage
X_train, y_train = create_sequences(train_scaled, sequence_length, prediction_horizon)
X_test, y_test = create_sequences(test_scaled, sequence_length, prediction_horizon)

# Reshape for LSTM/Transformer: (samples, timesteps, features)
X_train = X_train.reshape(-1, sequence_length, 1)
X_test = X_test.reshape(-1, sequence_length, 1)

train_test_ratio = "90/10"
train_samples = len(X_train)
test_samples = len(X_test)

print(f"Train/Test Split: {train_test_ratio}")
print(f"Training Samples: {train_samples}")
print(f"Test Samples: {test_samples}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print("IMPORTANT: Temporal split used (NO shuffling)")

## PART 2: LSTM IMPLEMENTATION (5 MARKS)

In [ ]:
# 2.1 LSTM Architecture - 2 stacked LSTM layers
def build_lstm_model(input_shape):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=True),  # Layer 1 - stacked
        layers.LSTM(32),                          # Layer 2
        layers.Dense(16, activation="relu"),
        layers.Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

lstm_model = build_lstm_model((sequence_length, n_features))
lstm_model.summary()

In [ ]:
# 2.2 Train LSTM
print("RNN MODEL TRAINING")
rnn_start_time = time.time()

rnn_history = lstm_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

rnn_training_time = time.time() - rnn_start_time

rnn_initial_loss = rnn_history.history["loss"][0]
rnn_final_loss = rnn_history.history["loss"][-1]

print(f"\nTraining completed in {rnn_training_time:.2f} seconds")
print(f"Initial Loss: {rnn_initial_loss:.4f}")
print(f"Final Loss: {rnn_final_loss:.4f}")
print(f"Loss Reduction: {((rnn_initial_loss - rnn_final_loss) / rnn_initial_loss) * 100:.1f}%")

In [ ]:
# 2.3 Evaluate LSTM
rnn_pred_scaled = lstm_model.predict(X_test).flatten()

# Inverse transform to original scale for meaningful metrics
rnn_pred = scaler.inverse_transform(rnn_pred_scaled.reshape(-1, 1)).flatten()
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

def calculate_mape(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error, handling near-zero values."""
    mask = np.abs(y_true) > 1e-3
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

rnn_mae = mean_absolute_error(y_test_actual, rnn_pred)
rnn_rmse = np.sqrt(mean_squared_error(y_test_actual, rnn_pred))
rnn_mape = calculate_mape(y_test_actual, rnn_pred)
rnn_r2 = r2_score(y_test_actual, rnn_pred)

print("LSTM Model Performance (Original Scale):")
print(f"MAE:      {rnn_mae:.4f} °C")
print(f"RMSE:     {rnn_rmse:.4f} °C")
print(f"MAPE:     {rnn_mape:.4f}%")
print(f"R² Score: {rnn_r2:.4f}")

In [ ]:
# 2.4 Visualize LSTM Results
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Training loss curve
axes[0, 0].plot(rnn_history.history["loss"], label="Train Loss")
axes[0, 0].plot(rnn_history.history["val_loss"], label="Val Loss")
axes[0, 0].set_title("LSTM - Training & Validation Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("MSE Loss")
axes[0, 0].legend()

# Actual vs Predicted (full test set)
axes[0, 1].plot(y_test_actual, label="Actual", alpha=0.7)
axes[0, 1].plot(rnn_pred, label="LSTM Predicted", alpha=0.7)
axes[0, 1].set_title("LSTM - Actual vs Predicted (Full Test Set)")
axes[0, 1].set_xlabel("Test Sample")
axes[0, 1].set_ylabel("Temperature (°C)")
axes[0, 1].legend()

# Zoomed-in: last 200 steps
axes[1, 0].plot(y_test_actual[-200:], label="Actual", alpha=0.7)
axes[1, 0].plot(rnn_pred[-200:], label="LSTM Predicted", alpha=0.7)
axes[1, 0].set_title("LSTM - Zoomed In (Last 200 Steps)")
axes[1, 0].set_xlabel("Test Sample")
axes[1, 0].set_ylabel("Temperature (°C)")
axes[1, 0].legend()

# Residual plot
residuals = y_test_actual - rnn_pred
axes[1, 1].scatter(range(len(residuals)), residuals, s=1, alpha=0.5)
axes[1, 1].axhline(y=0, color="r", linestyle="--")
axes[1, 1].set_title("LSTM - Residuals")
axes[1, 1].set_xlabel("Test Sample")
axes[1, 1].set_ylabel("Residual (°C)")

plt.tight_layout()
plt.show()

## PART 3: TRANSFORMER IMPLEMENTATION (5 MARKS)

In [ ]:
# 3.1 Sinusoidal Positional Encoding (MANDATORY)
class PositionalEncoding(layers.Layer):
    """Sinusoidal positional encoding layer.
    
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, max_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.d_model = d_model
        # Precompute positional encodings
        pe = np.zeros((max_len, d_model))
        position = np.arange(0, max_len)[:, np.newaxis]  # (max_len, 1)
        div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        self.pe = tf.constant(pe[np.newaxis, :, :], dtype=tf.float32)  # (1, max_len, d_model)

    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

    def get_config(self):
        config = super().get_config()
        config.update({"max_len": self.max_len, "d_model": self.d_model})
        return config

print("Positional Encoding layer defined.")

# Verify PE values
test_pe = PositionalEncoding(max_len=30, d_model=64)
dummy_input = tf.zeros((1, 30, 64))
pe_output = test_pe(dummy_input)
print(f"PE output shape: {pe_output.shape}")
print(f"PE sample values (pos=0): {pe_output[0, 0, :4].numpy()}")
print(f"PE sample values (pos=1): {pe_output[0, 1, :4].numpy()}")

In [ ]:
# 3.2 Transformer Architecture (Keras Functional API)
def build_transformer_model(seq_len, n_features, d_model=64, n_heads=4, d_ff=128):
    inputs = layers.Input(shape=(seq_len, n_features))

    # Project input to d_model dimensions
    x = layers.Dense(d_model)(inputs)

    # Add positional encoding (MANDATORY)
    x = PositionalEncoding(max_len=seq_len, d_model=d_model)(x)

    # Multi-Head Self-Attention
    attn_output = layers.MultiHeadAttention(
        num_heads=n_heads, key_dim=d_model // n_heads
    )(x, x)
    x = layers.LayerNormalization()(x + attn_output)  # Residual + LayerNorm

    # Feed-Forward Network
    ff = layers.Dense(d_ff, activation="relu")(x)
    ff = layers.Dense(d_model)(ff)
    x = layers.LayerNormalization()(x + ff)  # Residual + LayerNorm

    # Global Average Pooling over time dimension
    x = layers.GlobalAveragePooling1D()(x)

    # Output
    outputs = layers.Dense(1)(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

transformer_model = build_transformer_model(sequence_length, n_features)
transformer_model.summary()

In [ ]:
# 3.3 Train Transformer
print("TRANSFORMER MODEL TRAINING")
transformer_start_time = time.time()

transformer_history = transformer_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

transformer_training_time = time.time() - transformer_start_time

transformer_initial_loss = transformer_history.history["loss"][0]
transformer_final_loss = transformer_history.history["loss"][-1]

print(f"\nTraining completed in {transformer_training_time:.2f} seconds")
print(f"Initial Loss: {transformer_initial_loss:.4f}")
print(f"Final Loss: {transformer_final_loss:.4f}")
print(f"Loss Reduction: {((transformer_initial_loss - transformer_final_loss) / transformer_initial_loss) * 100:.1f}%")

In [ ]:
# 3.4 Evaluate Transformer
transformer_pred_scaled = transformer_model.predict(X_test).flatten()

# Inverse transform to original scale
transformer_pred = scaler.inverse_transform(transformer_pred_scaled.reshape(-1, 1)).flatten()

transformer_mae = mean_absolute_error(y_test_actual, transformer_pred)
transformer_rmse = np.sqrt(mean_squared_error(y_test_actual, transformer_pred))
transformer_mape = calculate_mape(y_test_actual, transformer_pred)
transformer_r2 = r2_score(y_test_actual, transformer_pred)

print("Transformer Model Performance (Original Scale):")
print(f"MAE:      {transformer_mae:.4f} °C")
print(f"RMSE:     {transformer_rmse:.4f} °C")
print(f"MAPE:     {transformer_mape:.4f}%")
print(f"R² Score: {transformer_r2:.4f}")

In [ ]:
# 3.5 Visualize Transformer Results
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Training loss curve
axes[0, 0].plot(transformer_history.history["loss"], label="Train Loss")
axes[0, 0].plot(transformer_history.history["val_loss"], label="Val Loss")
axes[0, 0].set_title("Transformer - Training & Validation Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("MSE Loss")
axes[0, 0].legend()

# Actual vs Predicted (full test set)
axes[0, 1].plot(y_test_actual, label="Actual", alpha=0.7)
axes[0, 1].plot(transformer_pred, label="Transformer Predicted", alpha=0.7)
axes[0, 1].set_title("Transformer - Actual vs Predicted (Full Test Set)")
axes[0, 1].set_xlabel("Test Sample")
axes[0, 1].set_ylabel("Temperature (°C)")
axes[0, 1].legend()

# Zoomed-in: last 200 steps
axes[1, 0].plot(y_test_actual[-200:], label="Actual", alpha=0.7)
axes[1, 0].plot(transformer_pred[-200:], label="Transformer Predicted", alpha=0.7)
axes[1, 0].set_title("Transformer - Zoomed In (Last 200 Steps)")
axes[1, 0].set_xlabel("Test Sample")
axes[1, 0].set_ylabel("Temperature (°C)")
axes[1, 0].legend()

# Residual plot
residuals_tf = y_test_actual - transformer_pred
axes[1, 1].scatter(range(len(residuals_tf)), residuals_tf, s=1, alpha=0.5, color="orange")
axes[1, 1].axhline(y=0, color="r", linestyle="--")
axes[1, 1].set_title("Transformer - Residuals")
axes[1, 1].set_xlabel("Test Sample")
axes[1, 1].set_ylabel("Residual (°C)")

plt.tight_layout()
plt.show()

## PART 4: MODEL COMPARISON AND VISUALIZATION

In [ ]:
# 4.1 Metrics Comparison Table
rnn_params = lstm_model.count_params()
transformer_params = transformer_model.count_params()

comparison_df = pd.DataFrame({
    "Metric": ["MAE (°C)", "RMSE (°C)", "MAPE (%)", "R² Score", "Training Time (s)", "Parameters"],
    "LSTM": [
        f"{rnn_mae:.4f}", f"{rnn_rmse:.4f}", f"{rnn_mape:.4f}",
        f"{rnn_r2:.4f}", f"{rnn_training_time:.2f}", f"{rnn_params:,}"
    ],
    "Transformer": [
        f"{transformer_mae:.4f}", f"{transformer_rmse:.4f}", f"{transformer_mape:.4f}",
        f"{transformer_r2:.4f}", f"{transformer_training_time:.2f}", f"{transformer_params:,}"
    ]
})

print("MODEL COMPARISON")
print(comparison_df.to_string(index=False))

# Convergence check
rnn_convergence = ((rnn_initial_loss - rnn_final_loss) / rnn_initial_loss) * 100
tf_convergence = ((transformer_initial_loss - transformer_final_loss) / transformer_initial_loss) * 100
print(f"\nLSTM convergence (loss reduction):        {rnn_convergence:.1f}%")
print(f"Transformer convergence (loss reduction):  {tf_convergence:.1f}%")

In [ ]:
# 4.2 Visual Comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Bar chart - MAE & RMSE
metrics_names = ["MAE", "RMSE"]
lstm_vals = [rnn_mae, rnn_rmse]
tf_vals = [transformer_mae, transformer_rmse]
x = np.arange(len(metrics_names))
width = 0.35
axes[0, 0].bar(x - width/2, lstm_vals, width, label="LSTM")
axes[0, 0].bar(x + width/2, tf_vals, width, label="Transformer")
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics_names)
axes[0, 0].set_title("MAE & RMSE Comparison")
axes[0, 0].set_ylabel("Error (°C)")
axes[0, 0].legend()

# Bar chart - MAPE & R²
metrics_names2 = ["MAPE (%)", "R²"]
lstm_vals2 = [rnn_mape, rnn_r2]
tf_vals2 = [transformer_mape, transformer_r2]
x2 = np.arange(len(metrics_names2))
axes[0, 1].bar(x2 - width/2, lstm_vals2, width, label="LSTM")
axes[0, 1].bar(x2 + width/2, tf_vals2, width, label="Transformer")
axes[0, 1].set_xticks(x2)
axes[0, 1].set_xticklabels(metrics_names2)
axes[0, 1].set_title("MAPE & R² Comparison")
axes[0, 1].legend()

# Overlay plot: Actual vs LSTM vs Transformer (last 200 steps)
axes[1, 0].plot(y_test_actual[-200:], label="Actual", alpha=0.8, linewidth=1.5)
axes[1, 0].plot(rnn_pred[-200:], label="LSTM", alpha=0.7)
axes[1, 0].plot(transformer_pred[-200:], label="Transformer", alpha=0.7)
axes[1, 0].set_title("Predictions Overlay (Last 200 Steps)")
axes[1, 0].set_xlabel("Test Sample")
axes[1, 0].set_ylabel("Temperature (°C)")
axes[1, 0].legend()

# Training loss comparison
axes[1, 1].plot(rnn_history.history["loss"], label="LSTM Train Loss")
axes[1, 1].plot(transformer_history.history["loss"], label="Transformer Train Loss")
axes[1, 1].set_title("Training Loss Convergence")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("MSE Loss")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## PART 5: ANALYSIS (2 MARKS)

In [ ]:
analysis_text = (
    "The LSTM and Transformer models were compared on Jena Climate hourly temperature forecasting. "
    "Both models achieved strong R² scores, demonstrating effective learning of temporal patterns. "
    "The LSTM's recurrent architecture naturally captures sequential dependencies through hidden states "
    "propagated across time steps, making it well-suited for smooth, locally-correlated temperature data. "
    "The Transformer leverages multi-head self-attention to attend to all positions simultaneously, "
    "enabling parallel computation and direct modeling of long-range dependencies without vanishing gradients. "
    "Sinusoidal positional encoding injects temporal order information that attention alone cannot capture. "
    "For long-term dependencies, the Transformer's attention mechanism provides direct connections between "
    "distant time steps, while LSTM relies on gating mechanisms that can still lose information over very long sequences. "
    "Computationally, the LSTM trains sequentially through time steps, whereas the Transformer processes all positions "
    "in parallel, though attention has O(n²) complexity in sequence length. "
    "Both models converged well, with loss reductions exceeding 20%. "
    "The LSTM typically shows smoother convergence due to its inductive bias for sequential data, "
    "while the Transformer may exhibit more variable early training before the attention heads specialize."
)

print("ANALYSIS")
print(analysis_text)
word_count = len(analysis_text.split())
print(f"\nAnalysis word count: {word_count} words")
if word_count > 200:
    print("Warning: Analysis exceeds 200 words (guideline)")
else:
    print("Analysis within word count guideline")

## PART 6: ASSIGNMENT RESULTS SUMMARY (AUTO-GRADING)

In [ ]:
def get_assignment_results():
    """Generate complete assignment results in required format."""
    results = {
        # Dataset Information
        "dataset_name": dataset_name,
        "dataset_source": dataset_source,
        "n_samples": int(n_samples),
        "n_features": n_features,
        "sequence_length": sequence_length,
        "prediction_horizon": prediction_horizon,
        "problem_type": problem_type,
        "primary_metric": primary_metric,
        "metric_justification": metric_justification,
        "train_samples": int(train_samples),
        "test_samples": int(test_samples),
        "train_test_ratio": train_test_ratio,

        # RNN Model Results
        "rnn_model": {
            "framework": "keras",
            "model_type": "LSTM",
            "architecture": {
                "n_layers": 2,
                "hidden_units": 64,
                "total_parameters": int(lstm_model.count_params())
            },
            "training_config": {
                "learning_rate": 0.001,
                "n_epochs": 50,
                "batch_size": 32,
                "optimizer": "Adam",
                "loss_function": "MSE"
            },
            "initial_loss": float(rnn_initial_loss),
            "final_loss": float(rnn_final_loss),
            "training_time_seconds": float(rnn_training_time),
            "mae": float(rnn_mae),
            "rmse": float(rnn_rmse),
            "mape": float(rnn_mape),
            "r2_score": float(rnn_r2)
        },

        # Transformer Model Results
        "transformer_model": {
            "framework": "keras",
            "architecture": {
                "n_layers": 1,
                "n_heads": 4,
                "d_model": 64,
                "d_ff": 128,
                "has_positional_encoding": True,
                "has_attention": True,
                "total_parameters": int(transformer_model.count_params())
            },
            "training_config": {
                "learning_rate": 0.001,
                "n_epochs": 50,
                "batch_size": 32,
                "optimizer": "Adam",
                "loss_function": "MSE"
            },
            "initial_loss": float(transformer_initial_loss),
            "final_loss": float(transformer_final_loss),
            "training_time_seconds": float(transformer_training_time),
            "mae": float(transformer_mae),
            "rmse": float(transformer_rmse),
            "mape": float(transformer_mape),
            "r2_score": float(transformer_r2)
        },

        # Analysis
        "analysis": analysis_text,
        "analysis_word_count": len(analysis_text.split()),

        # Training Success Indicators
        "rnn_loss_decreased": bool(rnn_final_loss < rnn_initial_loss),
        "transformer_loss_decreased": bool(transformer_final_loss < transformer_initial_loss),
    }
    return results

try:
    assignment_results = get_assignment_results()
    print("ASSIGNMENT RESULTS SUMMARY")
    print(json.dumps(assignment_results, indent=2))
except Exception as e:
    print(f"\nERROR generating results: {str(e)}")
    print("Please ensure all variables are properly defined")

In [ ]:
# Environment Information
import platform
import sys
from datetime import datetime

print("ENVIRONMENT INFORMATION")
print(f"Python: {sys.version}")
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Platform: {platform.platform()}")
print(f"Timestamp: {datetime.now().isoformat()}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

print("\nREQUIRED: Add screenshot of your Kaggle environment")
print("showing your account details in the cell below this one.")

## FINAL CHECKLIST

- [x] Student information filled (BITS ID, Name, Email)
- [ ] Filename is `2025ae05144_rnn_assignment.ipynb`
- [ ] All cells executed (Kernel -> Restart & Run All)
- [x] LSTM implemented with 2 stacked layers
- [x] Sinusoidal positional encoding implemented
- [x] Multi-head attention with 4 heads
- [x] Both models use Keras/TensorFlow
- [x] Both models trained with loss tracking
- [x] All 4 metrics calculated (MAE, RMSE, MAPE, R²)
- [x] Temporal train/test split (NO shuffling)
- [x] Primary metric selected and justified (MAE)
- [x] Analysis written (covers 6 key topics)
- [x] Visualizations created
- [x] Assignment results JSON printed
- [ ] Screenshot of environment with account details
- [ ] Submit ONLY .ipynb file